In [1]:
import json
import pandas as pd
import string
import re

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
from bert_score import score

import torch
from transformers import CLIPModel, CLIPTokenizer
import numpy as np

/home/gridsan/manderson/.conda/envs/skyscraper2/lib/python3.11/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/gridsan/manderson/.conda/envs/skyscraper2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# PATH_CKPT_CLIP14 = '/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14'
# CLIPModel.from_pretrained(PATH_CKPT_CLIP14)

In [2]:
def normalize(x):
    x = str(x).strip().lower()
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = re.sub(r"\s+", " ", x)
    return x

In [5]:
# Event detection

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/detection_teochat.csv"
df = pd.read_csv(csv_path)

y_true = df["ground_truth"] == 'yes'
y_pred = df["prediction"] == 'yes'

print("accuracy:", f"{accuracy_score(y_true, y_pred):.4f}")
print("precision:", f"{precision_score(y_true, y_pred, zero_division=0):.4f}")
print("recall:", f"{recall_score(y_true, y_pred, zero_division=0):.4f}")
print("f1:", f"{f1_score(y_true, y_pred, zero_division=0):.4f}")

accuracy: 0.4182
precision: 0.6858
recall: 0.1796
f1: 0.2847


In [6]:
# Event classification

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/classification_teochat.csv"
df = pd.read_csv(csv_path)

y_true = df["ground_truth"].apply(normalize)
y_pred = df["prediction"].apply(normalize)

print("accuracy:", f"{accuracy_score(y_true, y_pred):.4f}")
print("precision:", f"{precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print("recall:", f"{recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print("f1:", f"{f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")

#print(classification_report(y_true, y_pred, zero_division=0))

accuracy: 0.2898
precision: 0.6416
recall: 0.2898
f1: 0.3528


In [7]:
# Event description

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/description_teochat.csv"
df = pd.read_csv(csv_path)

# --- BLEU ---
smooth = SmoothingFunction().method1

bleu1, bleu2 = [], []

for _, row in df.iterrows():
    ref = [str(row["ground_truth"]).split()]
    hyp = str(row["prediction"]).split()

    bleu1.append(sentence_bleu(ref, hyp, weights=(1,0,0,0), smoothing_function=smooth))
    bleu2.append(sentence_bleu(ref, hyp, weights=(0.5,0.5,0,0), smoothing_function=smooth))
    
print("BLEU-1:", f"{sum(bleu1)/len(bleu1):.4f}")
print("BLEU-2:", f"{sum(bleu2)/len(bleu2):.4f}")
print()

# --- ROUGE ---
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

r1, r2, rL = [], [], []

for _, row in df.iterrows():
    scores = scorer.score(str(row["ground_truth"]), str(row["prediction"]))
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

print("ROUGE-1:", f"{sum(r1)/len(r1):.4f}")
print("ROUGE-2:", f"{sum(r2)/len(r2):.4f}")
print("ROUGE-L:", f"{sum(rL)/len(rL):.4f}")
print()

# --- METEOR ---
meteor_scores = []

for _, row in df.iterrows():
    ref = word_tokenize(str(row["ground_truth"]).lower())
    hyp = word_tokenize(str(row["prediction"]).lower())

    meteor_scores.append(meteor_score([ref], hyp))

print("METEOR:", f"{sum(meteor_scores)/len(meteor_scores):.4f}")
print()

# --- CIDEr ---
refs = {}
hyps = {}

for i, row in df.iterrows():
    refs[i] = [{"caption": str(row["ground_truth"])}]
    hyps[i] = [{"caption": str(row["prediction"])}]

tokenizer = PTBTokenizer()
refs_tok = tokenizer.tokenize(refs)
hyps_tok = tokenizer.tokenize(hyps)

cider_score, _ = Cider().compute_score(refs_tok, hyps_tok)
print("CIDEr:", f"{cider_score:.4f}")
print()

BLEU-1: 0.1422
BLEU-2: 0.0348

ROUGE-1: 0.1971
ROUGE-2: 0.0138
ROUGE-L: 0.1407

METEOR: 0.0996



PTBTokenizer tokenized 30591 tokens at 381552.01 tokens per second.
PTBTokenizer tokenized 22615 tokens at 329417.98 tokens per second.


CIDEr: 0.0221



In [8]:
# --- BERTScore ---
preds = df["prediction"].astype(str).tolist()
gts = df["ground_truth"].astype(str).tolist()

P, R, F1 = score(preds,
                 gts,
                 model_type="/home/gridsan/manderson/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594",
                 num_layers=9,
                 lang="en", 
                 verbose=True)

print("BERTScore Precision:", f"{P.mean().item():.4f}")
print("BERTScore Recall:", f"{R.mean().item():.4f}")
print("BERTScore F1:", f"{F1.mean().item():.4f}")

calculating scores...
computing bert embedding.


100%|██████████| 24/24 [00:09<00:00,  2.47it/s]


computing greedy matching.


100%|██████████| 14/14 [00:03<00:00,  3.51it/s]


done in 13.78 seconds, 63.85 sentences/sec
BERTScore Precision: 0.4940
BERTScore Recall: 0.4651
BERTScore F1: 0.4788


In [9]:
# --- CLIP Score ---

device = "cuda" if torch.cuda.is_available() else "cpu"

# load model + tokenizer
PATH_CKPT_CLIP14 = '/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14'
model = CLIPModel.from_pretrained(PATH_CKPT_CLIP14)
tokenizer = CLIPTokenizer.from_pretrained(PATH_CKPT_CLIP14)
model = model.to(device)
model.eval()

# get text data
preds = df["prediction"].astype(str).tolist()
gts = df["ground_truth"].astype(str).tolist()

# compute similarity
scores = []

with torch.no_grad():
    for pred, gt in zip(preds, gts):
        tokens = tokenizer([pred, gt], padding=True, truncation=True, return_tensors="pt")
        tokens = {k: v.to(device) for k, v in tokens.items()}

        feats = model.get_text_features(**tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        sim = (feats[0] @ feats[1]).item()
        scores.append(sim)

clipscore = sum(scores) / len(scores)

print(f"CLIPScore: {clipscore:.4f}")

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.


CLIPScore: 0.5179


In [5]:
csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/grounding_teochat-sample.csv"
json_path = "/home/gridsan/manderson/skyscraper-s2/data/skyscraper_gdelt_sentinel/vqa/teochat_sentinel_event_grounding_val-sample.json"

def parse_gt(s):
    dates = re.findall(r"\d{4}-\d{2}-\d{2}|\d{8}", str(s))

    norm = []
    for d in dates:
        if len(d) == 8 and "-" not in d:
            d = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
        norm.append(d)

    if len(norm) < 2:
        raise ValueError(f"could not parse GT dates: {s}")

    return norm[0], norm[-1]

def parse_pred(s):
    s = str(s)

    # find both YYYY-MM-DD and YYYYMMDD
    dates = re.findall(r"\d{4}-\d{2}-\d{2}|\d{8}", s)

    # normalize YYYYMMDD -> YYYY-MM-DD
    norm = []
    for d in dates:
        if len(d) == 8 and "-" not in d:
            d = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
        norm.append(d)

    if len(norm) < 2:
        raise ValueError("could not parse 2 dates")

    # if model output many dates, use first and last
    return norm[0], norm[-1]

def iou_discrete(pred_s, pred_e, gt_s, gt_e):
    pred_set = set(range(pred_s, pred_e + 1))
    gt_set = set(range(gt_s, gt_e + 1))
    inter = len(pred_set & gt_set)
    union = len(pred_set | gt_set)
    return inter / union if union > 0 else 0.0

df = pd.read_csv(csv_path)

with open(json_path, "r") as f:
    data = json.load(f)

id_to_timestamps = {item["id"]: item["timestamp"] for item in data}

fail_count = 0
start_correct = []
end_correct = []
start_abs_err = []
end_abs_err = []
ious = []

for _, row in df.iterrows():
    ex_id = int(row["id"])
    timestamps = id_to_timestamps[ex_id]
    #print(timestamps)

    gt_start, gt_end = parse_gt(row["ground_truth"])
    #print(gt_start, gt_end)
    gt_s_idx = timestamps.index(gt_start)
    gt_e_idx = timestamps.index(gt_end)
    #print(gt_s_idx, gt_e_idx)

    try:
        pred_start, pred_end = parse_pred(row["prediction"])
        #print(pred_start, pred_end)
        pred_s_idx = timestamps.index(pred_start)
        pred_e_idx = timestamps.index(pred_end)
        #print(pred_s_idx, pred_e_idx)

        start_correct.append(int(pred_s_idx == gt_s_idx))
        end_correct.append(int(pred_e_idx == gt_e_idx))
        start_abs_err.append(abs(pred_s_idx - gt_s_idx))
        end_abs_err.append(abs(pred_e_idx - gt_e_idx))
        ious.append(iou_discrete(pred_s_idx, pred_e_idx, gt_s_idx, gt_e_idx))

    except:
        fail_count += 1
        max_err = len(timestamps) - 1
        start_correct.append(0)
        end_correct.append(0)
        start_abs_err.append(max_err)
        end_abs_err.append(max_err)
        ious.append(0.0)

n = len(df)

print(f"Failures: {fail_count}/{n} ({fail_count/n:.4f})")
print(f"Accuracy start: {np.mean(start_correct):.4f}")
print(f"Accuracy end:   {np.mean(end_correct):.4f}")
print(f"MAE start:      {np.mean(start_abs_err):.4f}")
print(f"MAE end:        {np.mean(end_abs_err):.4f}")
print(f"mIOU:           {np.mean(ious):.4f}")

Failures: 35/880 (0.0398)
Accuracy start: 0.2398
Accuracy end:   0.3205
MAE start:      1.9159
MAE end:        1.8307
mIOU:           0.3926
